<a href="https://colab.research.google.com/github/vivacitylabs/data-toolkit/blob/master/notebooks/occupancy_bulk_download_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Occupancy - Bulk Download Generator



## Generate a csv file of zonal occupancy data over multiple days

This notebook is a tool to access VivaCity data via the API. It is aimed as an **interim solution** while we're working on new dashboard developments. You can contact customer support if you have any issues (support@vivacitylabs.com) or raise a ticked on the [Customer Help Portal](https://vivacitylabs.atlassian.net/servicedesk/customer/portal/16).

#### How it works

This notebook will run you through all the necessary steps and will save a csv file locally or in your Google Drive.

You will need to fill in a few details and then hit the run button (▶) next to the code cells.

If you want to make changes to the code and save them, you will first need to save a copy of this notebook to your Google Drive.



**What you will need**

- VivaCity API login credentials
- Sensors and their zones you want to download data for



ℹ  Note the notebook only works for zones that have [occupancy](https://vivacitylabs.customerly.help/vivacity-dashboard/occupancy) enabled


#### Output format

You will receive occupancy measures in selected time buckets in the following format. The mean objects and mean stopped objects per second are derived from the `total_count` and `stopped_count` columns. For an explanation on those, read the API documentation [here](https://beta.docs.api.vivacitylabs.com/).


| Start Date | Start time | End time | zone_id |  zone_description | 	class |	total_count|	stopped_count |	mean_objects_second |	mean_stopped_objects_second |
|:---------:|:---------:|:---------:|:---------:|:---------:|:---------:|:---------:|:---------:|:---------:|:---------:|
| 2022-10-17 |00:00:00	| 00:15:00 |	2245 | S4 GrandDrive - Zone 1 (2245) | car	|{'1': 20}|{}|	0.0056 |	0.0000	|



## Stage 1: Getting Started
Let's begin by importing the packages we'll need and creating some useful functions!

Hit the run button (▶) in the top left corner.

In [ ]:
#@title { vertical-output: false, display-mode: "form" }
#@markdown **Code cell:** Run this to import functions and connect to Google Drive
import requests
import getpass
import json
from datetime import date, datetime, timedelta
import pytz
import pandas as pd
import csv
import time
from IPython.display import Markdown, display
def printmd(string):
    display(Markdown(string))
from ipywidgets import interact, interactive, fixed, interact_manual, Layout, Box
import ipywidgets as widgets
import warnings
warnings.filterwarnings('ignore')

def get_date_range(start_date, end_date):
    start_dates = []
    end_dates = []

    start_date = datetime.fromisoformat(start_date)
    end_date = datetime.fromisoformat(end_date)
    while True:
        start_dates.append(start_date.strftime('%Y-%m-%dT%H:%M:%S.000Z'))
        end_dates.append((start_date+timedelta(days=1)).strftime('%Y-%m-%dT%H:%M:%S.000Z'))
        start_date = start_date+timedelta(days=1)
        if start_date > end_date:
            break
    date_range = list(zip(start_dates, end_dates))
    return date_range

## Stage 2: Data Import
At the end of this process we will requested data from the API for the dates in the range that you have determined in the **Zone Details** step.

The resulting JSON responses will then be converted into a data table in the **Data Processing** step.

Authentication is handled at this stage.

### Authentication
You'll need a VivaCity API key. You can create one in the [User Settings page](https://admin.vivacitylabs.com/settings/api-access). If you have any issues, please contact customer support (support@vivacitylabs.com).

1.   Insert your API key into the box on the right.
2.   Hit the run button (▶).


In [ ]:
#@title  { run: "auto", vertical-output: true, display-mode: "form" }
#@markdown **Code cell:** Insert your API key, then run

api_key = "api-key" #@param {type:"string"}

headers = {
    'Accept': 'application/json',
    'x-vivacity-api-key': api_key
}

### Retrieve available sensors and zones

We'll now get all sensors and their zones the API user has access to (which have occupancy enabled).

In [ ]:
#@title { vertical-output: false, display-mode: "form" }
#@markdown **Code cell:** Run this to retrieve zones available to you from the API

print("Requesting zone metadata ...")
api_url_base = 'https://api.vivacitylabs.com'

# Use a Session with retries to handle the occasional chunked-encoding hiccup on large metadata responses
session = requests.Session()
session.headers.update(headers)

zone_metadata = {}
cursor = None
page_size = 1000
while True:
    params_meta = {'page_size': page_size}
    if cursor:
        params_meta['cursor'] = cursor
    last_err = None
    for attempt in range(5):
        try:
            r = session.get(f'{api_url_base}/zone/metadata', params=params_meta, timeout=120)
            break
        except requests.exceptions.RequestException as e:
            last_err = e
            time.sleep(2 ** attempt)
    else:
        raise RuntimeError(f"Failed to fetch zone metadata after retries: {last_err}")

    if r.status_code == 401:
        print("\n!Error: Can't access the data. Check your API key, or ask customer support if your user is set up correctly.\n")
        raise SystemExit
    r.raise_for_status()
    body = r.json()

    # Endpoint returns either a flat dict {zone_id: {...}} or a paged dict {data: {...}, next_cursor: ...}
    if isinstance(body, dict) and 'data' in body and isinstance(body['data'], dict):
        zone_metadata.update(body['data'])
        cursor = body.get('next_cursor') or body.get('cursor')
        if not cursor:
            break
    else:
        zone_metadata.update(body)
        break

# Filter to occupancy-enabled zones
dict_zones = {"zone_id": [], "zone_name": [], "zone_description": []}
for zone_id, z in zone_metadata.items():
    if not isinstance(z, dict):
        continue
    if not z.get('is_occupancy'):
        continue
    dict_zones["zone_id"].append(zone_id)
    dict_zones["zone_name"].append(z.get('name', ''))
    dict_zones["zone_description"].append(z.get('description', ''))

df_hard = pd.DataFrame.from_dict(dict_zones).drop_duplicates().reset_index(drop=True)

# Build a human-readable dropdown label: "name - description (zone_id)" (omit description if empty)
def _label(row):
    if row['zone_description']:
        return f"{row['zone_name']} - {row['zone_description']} ({row['zone_id']})"
    return f"{row['zone_name']} ({row['zone_id']})"
df_hard["zones_dropdown"] = df_hard.apply(_label, axis=1)
df_hard = df_hard.sort_values("zones_dropdown").reset_index(drop=True)

print(f"Done. {len(df_hard)} occupancy-enabled zones available.")

### Select zones and dates
Choose one or more zones, the classes you want data for, the date period and time bucket (15min, 1h, 24h).

**Note:** The sensor name is derived from countline names (eg. countline name: S4_harleyRd_crossing => extracted sensor name: S4_harleyRd). Sometimes countlines are not named consistently resulting in odd sensor names.

In [ ]:
#@title { vertical-output: false, display-mode: "form" }
#@markdown **Code cell:** Run this and then make your selection below
box_layout = Layout(display='flex', flex_flow='column', align_items='stretch', border=None, width='28%')

start_date_input = widgets.DatePicker(description="Start date",layout=Layout(width='55%'))
end_date_input = widgets.DatePicker(description="End date",layout=Layout(width='55%'))
timezone = widgets.Dropdown(options=['Europe/London', "Europe/Berlin", "Australia/Sydney"],description="Timezone",layout=Layout(width='55%'))
timebucket_input = widgets.Dropdown(options=['15m', "1h", "24h"],description="Time bucket",layout=Layout(width='55%'))
zones_input = widgets.SelectMultiple(
    options=df_hard["zones_dropdown"].unique(),
    description='Zones',
    disabled=False,
    layout=Layout(width='auto', height='170px'))
class_input = widgets.SelectMultiple(
    options=[ "cyclist", "motorbike", "car", "pedestrian", "taxi", "van", "minibus", "bus", "rigid", "truck", "emergency_car", "emergency_van", "fire_engine", "escooter"],
    description='Class',  disabled=False,
    layout=Layout(width='55%', height='235px')
)

items = [start_date_input, end_date_input, timezone,timebucket_input, class_input, zones_input]
box = Box(children=items, layout=box_layout)
printmd("**Select date period and zones**")
printmd("Hold  `Ctrl`  to select multiple classes or zones")
print("")
box

### Getting the data


We now query occupancy data from the API.

The output will tell you how many requests are made and what the progress is.


In [ ]:
#@title { vertical-output: false, display-mode: "form" }
#@markdown **Code cell:** Run this to set the API request parameters and check your selection again
params = {}
params['zone_ids'] = df_hard[df_hard["zones_dropdown"].isin(zones_input.value)]["zone_id"].to_list()
params['classes'] = list(class_input.value)
print(params['classes'])
params["time_bucket"] = timebucket_input.value
params['fill_nulls'] = "true"

#convert local datetime to UTC datetime
start_date_utc = str(pd.to_datetime(start_date_input.value).tz_localize(timezone.value).astimezone(pytz.utc))
end_date_utc = str(pd.to_datetime(end_date_input.value).tz_localize(timezone.value).astimezone(pytz.utc))

#check if dates are in correct order
if start_date_input.value > end_date_input.value:
  print("Start date is after end date, please correct your date selection")
else:
  date_range = get_date_range(start_date_utc, end_date_utc)
  printmd("**Check your selection:**\n")
  print("Dates:", start_date_input.value, "to", end_date_input.value, "\nClass:", list(class_input.value),
        "\nZones:", list(zones_input.value) , "\nTimebucket:", timebucket_input.value)

In [ ]:
#@title { vertical-output: false, display-mode: "form" }
#@markdown **Code cell:** Run this to request data from the API (can take a bit)

# Rate limits: 300 requests/minute and 7200 requests/hour per source IP.
# We stay safely below both with sliding-window tracking and a small minimum gap between requests.
from collections import deque

max_requests_per_minute = 290
max_requests_per_hour = 7100
min_request_interval = 60 / max_requests_per_minute

minute_window = deque()
hour_window = deque()

def wait_for_rate_limit():
    """Block until we're safely under both per-minute and per-hour API limits."""
    now = time.time()
    while minute_window and now - minute_window[0] > 60:
        minute_window.popleft()
    while hour_window and now - hour_window[0] > 3600:
        hour_window.popleft()

    if len(minute_window) >= max_requests_per_minute:
        sleep_time = 60 - (now - minute_window[0]) + 0.05
        if sleep_time > 0:
            time.sleep(sleep_time)
    if len(hour_window) >= max_requests_per_hour:
        sleep_time = 3600 - (time.time() - hour_window[0]) + 0.05
        if sleep_time > 0:
            print(f"Hourly rate limit reached, sleeping {sleep_time:.0f}s ...")
            time.sleep(sleep_time)

    if minute_window:
        time_since_last = time.time() - minute_window[-1]
        if time_since_last < min_request_interval:
            time.sleep(min_request_interval - time_since_last)

data = []
for i, date in enumerate(date_range):
    params["from"] = date[0]
    params["to"] = date[1]

    wait_for_rate_limit()

    # Retry transient failures (429, 5xx) with exponential backoff
    response = None
    for attempt in range(5):
        request_time = time.time()
        minute_window.append(request_time)
        hour_window.append(request_time)
        try:
            response = requests.get(
                'https://api.vivacitylabs.com/zone/occupancy',
                params=params, headers=headers, timeout=60
            )
        except requests.exceptions.RequestException as e:
            print(f"Request error on attempt {attempt+1}: {e}")
            time.sleep(2 ** attempt)
            continue
        if response.status_code == 429:
            retry_after = int(response.headers.get('Retry-After', 2 ** attempt))
            print(f"Rate limited (429). Sleeping {retry_after}s before retry.")
            time.sleep(retry_after)
            continue
        if 500 <= response.status_code < 600:
            print(f"Server error {response.status_code} on attempt {attempt+1}. Retrying...")
            time.sleep(2 ** attempt)
            continue
        break

    print(f"{i+1}/{len(date_range)}: {response.status_code if response is not None else 'no-response'} {response.reason if response is not None else ''}")

    if response is None:
        print(f"Skipped {date[0]} to {date[1]} after retries failed")
        continue

    if response.status_code == 200:
        occ_json = response.json()
        occ_dict = {"zone_id":[], "from":[], "to":[], "class":[], "total_count":[], "stopped_count":[]}

        for zone, items in occ_json.items():
            for time_bucket in items:
                for _class in time_bucket['class_occupancies'].keys():
                    occ_dict["zone_id"].append(zone)
                    occ_dict["from"].append(datetime.strptime(time_bucket['from'],'%Y-%m-%dT%H:%M:%S.000Z'))
                    occ_dict["to"].append(datetime.strptime(time_bucket['to'],'%Y-%m-%dT%H:%M:%S.000Z'))
                    occ_dict["class"].append(_class)
                    occ_dict["total_count"].append(time_bucket['class_occupancies'][_class]["total"])
                    occ_dict["stopped_count"].append(time_bucket['class_occupancies'][_class]["stopped"])

        if any(len(v) > 0 for v in occ_dict.values()):
            data.append(pd.DataFrame.from_dict(occ_dict))
    elif response.status_code == 204:
        # No content for this period - normal when there is no data
        pass
    else:
        print(f"Data missing for {date[0].split('T')[0]} to {date[1].split('T')[0]}: {response.text[:200]}")

if data:
    data = pd.concat(data, axis=0, ignore_index=True)
    print(f"\nCollected {len(data)} rows.")
else:
    data = pd.DataFrame()
    print("\nNo data was collected.")

## Stage 3: Data Processing
Now we process the raw data output, this might take a bit depending on the size of your data set.

In [ ]:
#@title  { vertical-output: false, display-mode: "form" }
#@markdown **Code cell:** Run to process data

if timebucket_input.value=="15m":
  seconds = 15*60
elif timebucket_input.value== "1h":
  seconds = 60*60
else:
  seconds = 24*60*60

#calculate mean objects per second
mean_objects_second = []
for i in range(len(data)):
  sum = 0
  for items in data.iloc[i]["total_count"].items():
    sum +=int(items[0])*items[1]
  mean_objects_second.append(round(sum/seconds,4))
data["mean_objects_second"] = mean_objects_second

mean_stopped_second = []
for i in range(len(data)):
  sum = 0
  for items in data.iloc[i]["stopped_count"].items():
    sum +=int(items[0])*items[1]
  mean_stopped_second.append(round(sum/seconds,4))
data["mean_stopped_objects_second"] = mean_stopped_second

#convert back to local datetime
data["Date"] = [ str(pd.to_datetime(i).tz_localize('utc').astimezone(timezone.value).date()) for i in data["from"]]
data["Start time"] = [ str(pd.to_datetime(i).tz_localize('utc').astimezone(timezone.value).time()) for i in data["from"]]
data["End time"] = [ str(pd.to_datetime(i).tz_localize('utc').astimezone(timezone.value).time()) for i in data["to"]]

#merge in zone name
data = pd.merge(data, df_hard[["zone_id", "zones_dropdown"]], left_on="zone_id", right_on="zone_id", how='left')

#rename and reorder columns
data = data.rename(columns={"zones_dropdown":"zone_description"})
export = data[[ 'Date','Start time', 'End time','zone_id', 'zone_description','class', 'total_count', 'stopped_count',
       'mean_objects_second', 'mean_stopped_objects_second' ]]

## Stage 4: Data Export
Now let's write this to a .csv file. You can either save the file locally (it will show in your Downloads folder) or save it to a Google Drive.


* **Local Downloads Folder:** This might not work if your browser or computer blocking downloads.
* **Google Drive:** If you want to save it in Google Drive, you will be asked for permission to connect to your Google Account.

In [ ]:
#@title  { vertical-output: true, display-mode: "form" }
#@markdown Select where to save the csv file
download_location = "Local folder" #@param [ "Local folder", "Google Drive"]
#@markdown Name your file
filename = "occupancy-test" #@param {type:"string"}
#@markdown Hit run (>)

if download_location == "Local folder":
  from google.colab import files
  export.to_csv(filename + ".csv", index = False)
  files.download(filename + ".csv")
else:
  from google.colab import drive
  drive.mount('/content/drive')
  path = '/content/drive/My Drive/'
  export.to_csv(path + filename +".csv", index = False)